In [2]:
import pandas as pd

In [3]:
df_pedro = pd.read_csv('../data/processed/datos_pedro.csv', sep='|', index_col=0)
df_mili = pd.read_csv('../data/processed/pib_vs_calidad_vida.csv', sep=',', index_col=0).reset_index()
df_manu = pd.read_csv('../data/interim/alquiler_venta_ccaa.csv', sep=',', index_col=0)

# Renombramos columnas de comunidades autónomas y año para realizar el merge en ellas
df_mili = df_mili.rename(columns = {'Comunidad Autónoma':'com_aut'})
df_manu = df_manu.rename(columns = {'Año':'año', 'Comunidad Autonoma':'com_aut'})


In [4]:
df_p = df_pedro[(df_pedro["año"] > 2008) & (df_pedro["año"] < 2023)]

In [5]:
df_p[df_p['crim'].isna()]

,año,com_aut,pib,pob,sui,nat,paro,ing,pobr,fum,alc,obes,fyv,crim,homi,sati
171,2009,Andalucía,145802256,8244.5,10.87,11.48,17.73,9437.0,NaN,20.93,1.29,19.67,80.82,NaN,NaN,NaN
172,2009,Aragón,33814117,1344.5,7.21,9.72,7.29,11963.0,NaN,17.76,2.98,15.75,86.35,NaN,NaN,NaN
173,2009,Asturias,22366414,1076.4,9.47,7.63,8.50,12412.0,NaN,19.24,3.87,19.12,72.70,NaN,NaN,NaN
174,2009,Balears,26770420,1078.1,8.16,11.17,10.16,11909.0,NaN,20.54,1.70,16.35,21.48,NaN,NaN,NaN
175,2009,Canarias,39505370,2034.2,9.72,9.32,17.25,9416.0,NaN,20.69,1.55,17.50,65.47,NaN,NaN,NaN
176,2009,Cantabria,12811684,586.8,3.98,9.58,7.16,11736.0,NaN,18.88,0.89,13.68,59.69,NaN,NaN,NaN
177,2009,Castilla Y León,54452146,2547.6,7.87,8.01,9.62,10852.0,NaN,18.98,3.59,16.82,83.80,NaN,NaN,NaN
178,2009,Castilla - La Mancha,39355657,2075.9,7.72,10.75,11.67,9530.0,NaN,20.60,2.37,14.40,80.40,NaN,NaN,NaN
179,2009,Cataluña,200745296,7447.3,5.84,11.44,8.89,13048.0,NaN,20.91,1.09,14.20,77.65,NaN,NaN,NaN
180,2009,Comunitat Valenciana,101989994,4984.4,7.65,10.51,11.99,10310.0,NaN,19.88,2.06,16.66,80.60,NaN,NaN,NaN


In [6]:
MAP = {'Andalucia':'Andalucía',
       'Aragon':'Aragón',
       'Baleares':'Balears' ,
       'Castilla-La Mancha':'Castilla - La Mancha',
       'Comunidad Valenciana':'Comunitat Valenciana',
       'La Rioja':'Rioja',
       'Euskadi': 'País Vasco'}
for i,j in MAP.items():
    df_manu["com_aut"] = df_manu["com_aut"].replace(i,j)

In [7]:
def merge_datasets(df_left:pd.DataFrame, df_right:pd.DataFrame, on1_left:str, on1_right:str, on2_left:str, on2_right:str):
    '''
    Función para hacer merge de dos datasets con las mismas referencias ordenadas pero con variaciones en el nombre. Fuerza el cambio de nombres para hacer merge.
    '''
    values_left = df_left[on1_left].sort_values().unique()
    values_right = df_right[on1_right].sort_values().unique()
    dicc = {values_right[i]: values_left[i] for i in range(len(values_right))}
    df_right[on1_right] = df_right[on1_right].map(dicc)

    df_right = df_right.rename(columns={on1_right: on1_left})
    df_right = df_right.rename(columns={on2_right: on2_left})

    return pd.merge(df_left, df_right, 'outer', [on1_left, on2_left])

In [8]:
df_pedro_mili = merge_datasets(df_pedro, df_mili, 'com_aut', 'com_aut', 'año', 'año')

In [9]:
df_final = merge_datasets(df_pedro_mili , df_manu, 'com_aut', 'com_aut', 'año', 'año')

In [11]:
df = df_final.copy()
df = df[(df["año"] > 2008) & (df["año"] < 2023)]

In [12]:
df.to_csv("../data/processed/dataset_sucio.csv", index=False)

**La importancia incial se mide clasificándolas en target/directora (0), agrupación importante (1), agrupación interesante (2), agrupación secundaria (3)**  
0 (directoras): estructuran el análisis temporal y territorial.  
1 (núcleo): variables económicas y de bienestar directamente relacionadas con el PIB.  
2 (impacto social): reflejan consecuencias o mediadores del nivel económico.  
3 (contexto): variables de control y descriptivas.  

| Variable | Descripción | Tipo de variable | Importancia inicial |
|--------|------------|-----------------|---------------------|
| año | Año de referencia de los datos. | Numérica discreta | 0 |
| com_aut | Comunidad Autónoma de España. | Categórica nominal | 0 |
| pib | Producto Interior Bruto. Valor económico agregado. | Numérica continua | 0 |
| pob | Población total de la comunidad. | Numérica discreta | 3 |
| sui | Tasa de suicidios. Defunciones por suicidio por 100.000 habitantes. | Numérica continua | 2 |
| nat | Tasa de natalidad. Nacimientos por cada 1.000 habitantes. | Numérica continua | 2 |
| paro | Tasa de desempleo. Parados sobre población activa. | Numérica continua | 1 |
| ing | Ingresos medios por persona adulta. | Numérica continua | 1 |
| pobr | Tasa de riesgo de pobreza. | Numérica continua | 1 |
| fum | Porcentaje de población adulta fumadora. | Numérica continua | 3 |
| alc | Porcentaje de población adulta con consumo de riesgo de alcohol. | Numérica continua | 3 |
| obes | Porcentaje de población adulta con obesidad. | Numérica continua | 3 |
| fyv | Porcentaje de población con consumo diario de frutas y verduras. | Numérica continua | 3 |
| crim | Tasa de criminalidad. Infracciones penales por 1.000 habitantes. | Numérica continua | 2 |
| homi | Tasa de homicidios. Homicidios y asesinatos por 100.000 habitantes. | Numérica continua | 3 |
| sati | Nivel de satisfacción general con la vida. | Numérica continua | 1 |
| pib_pc | PIB per cápita a precios de mercado. | Numérica continua | 1 |
| renta_pc | Renta disponible bruta de los hogares per cápita. | Numérica continua | 1 |
| poblacion | Número total de habitantes por comunidad y año. | Numérica discreta | 3 |
| gasto_elevado_vivienda(%) | Porcentaje de población con gasto elevado en vivienda. | Numérica continua | 2 |
| falta_espacio_vivienda(%) | Porcentaje de población que vive en viviendas con falta de espacio. | Numérica continua | 3 |
| retrasos_pagos(%) | Porcentaje de población con retrasos en pagos de vivienda o suministros. | Numérica continua | 2 |
| renta_media | Renta media por unidad de consumo. | Numérica continua | 1 |
| renta_mediana | Renta mediana por unidad de consumo. | Numérica continua | 1 |
| riesgo_pobreza(%) | Porcentaje de población en riesgo de pobreza. | Numérica continua | 1 |
| dificultad_fin_mes(%) | Porcentaje de población con dificultad para llegar a fin de mes. | Numérica continua | 2 |
| desigualdad_ing(S80/S20) | Cociente de desigualdad entre el 20% más rico y el 20% más pobre. | Numérica continua | 1 |
| inc_gastos_imprevistos(%) | Porcentaje de población que no puede afrontar gastos imprevistos. | Numérica continua | 2 |
| tasa_criminalidad | Delitos registrados por cada 1.000 habitantes. | Numérica continua | 2 |
| precio_medio_anual_eur_m2_venta | Precio medio anual de vivienda en venta por m² (€). | Numérica continua | 2 |
| precio_medio_anual_eur_m2_alquiler | Precio medio anual de vivienda en alquiler por m² (€). | Numérica continua | 2 |


In [ ]:
Hipotesis 